In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )

Rdair=Co.Rdair()


In [ ]:
%%time

use_new_function = True

if use_new_function == True :
    print( 'Using new read case')
    case1 = 'cam77_dyamond1_prod1'  #'c153_topfix_ne240pg3_FMTHIST_xic_x03'
    A1 = futi.read_case( case=case1 , nsteps = 31*8 )
    
    plev=A1.plev
    lat=A1.lat
    lon=A1.lon
    
    zlev=A1.zlev 
    
    reso='14km'

    htopo=A1.htopo

    epwp_1 = A1.epwp
    upwp_1 = A1.upwp
    vpwp_1 = A1.vpwp
    rho_epwp_1 = A1.rho_epwp
    
    u_1 = A1.u
    v_1 = A1.v
    te_1 = A1.te
    pmid_1,pint_1 = A1.pmid, A1.pint
    ze_1, zo_1 = A1.ze, A1.zo
    zeta_1 = A1.zeta  
    RhoProxy=A1.RhoProxy

    nt,nz,ny,nx = np.shape( u_1 )






In [ ]:
##
from Regridder import VertRegridFlexLL as vrg


#plevs = np.asarray( [100_000.,50_000.,10_000., 5_000., 2_500. , 1_000., 500., 100. ] )
ztarg = np.linspace( 5_000.,28_000.,num=24 )
#pint_Dst = plevs[None, :, None, None]

ztarg3 = np.tile(ztarg[None, :, None, None], (nt, 1, ny, nx) )
A1['dycore']='MPAS'
if A1.dycore == 'MPAS':
    zo_1 = np.tile(zo_1[None, :, :, :], (nt, 1, 1, 1) )

In [ ]:
print( zo_1.shape )
RhoProxy3 = np.tile(RhoProxy[None, :, None, None], (nt, 1, ny, nx) )

#rho_1 = pmid_1 / (Rdair * te_1 )
#rho_epwp_1 = (rho_1/RhoProxy3) * epwp_1


In [ ]:
print(ztarg3.shape)
print(ztarg)

In [ ]:
DstTZHkey = 'tzyx'

rho_epwp_1_z = vrg.VertRG( a_x  = rho_epwp_1 ,
                        zSrc = zo_1 ,
                        zDst = ztarg3 ,
                        Gridkey =DstTZHkey ,
                        kind = 'linear' ) #linea


In [ ]:

trunc_k, sigma=10, 4.5
#trunc_k, sigma=20, 7.5

htopo_sm = gaussian_filter(  htopo , sigma=sigma,
                           truncate=((trunc_k-1)/2)/sigma, mode="nearest")

mask=htopo_sm <= 1.0

z=np.argmin( np.abs( zlev - 20_000. ) )

zz=np.argmin( np.abs( ztarg - 20_000. ) )

title_ = f'Z={zlev[z]/1000.:.0f} km '     #        camsnap-yaaaa.h.2004-06-15-21600.nc'


#flev=[0.0001,0.0002,0.0005,  0.001,0.002,0.005,  0.01,0.02,0.05,  0.1,0.2,0.5,   1.0  ]
flev=[0.0001,0.0002,0.0005,  0.001,0.0015,0.002,0.003,0.004,0.005,0.006,0.008,    0.01,0.015,0.02,0.03,0.04,0.05,0.06,  0.1,0.2,0.5,   1.0  ]
flev=0.1*np.asarray( flev )
print(flev)
cmapN='gist_ncar'

cmap = plt.cm.bwr  #gist_ncar  # Or any other colormap
cmap = plt.cm.gist_ncar # .plasma #gist_ncar  #gist_ncar  # Or any other colormap
norm = mcolors.BoundaryNorm(boundaries=flev, ncolors=cmap.N, clip=False)

topocolor='white' #'black'
topocolor='black'


fig,axs=plt.subplots( 1, 2, figsize=(30,6) )

ax=axs[0]
c = ax.contourf( lon, lat, np.average(rho_epwp_1[:,z,:,:],axis=0) ,levels=flev, cmap=cmap , norm=norm, extend='both'  )
#c = ax.contourf( lon, lat, np.average(epwp_1[:,z,:,:],axis=0) ,levels=21, cmap=cmap , extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
ax.set_title( f'Resolved GW {r"$\tau$"} in {case1}: {reso} sim.')
plt.colorbar( c )

ax=axs[1]
c = ax.contourf( lon, lat, np.average(rho_epwp_1_z[:,zz,:,:],axis=0),levels=flev, cmap=cmap , norm=norm, extend='both'  )
#c = ax.contourf( lon, lat, np.average(rho_epwp_1[:,z,:,:],axis=0) *mask,levels=flev, cmap=cmap , norm=norm, extend='both'  )
#c = ax.contourf( lon, lat, np.average(epwp_1[:,z,:,:],axis=0) ,levels=21, cmap=cmap , extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
ax.set_title( f'Resolved GW {r"$\tau$"} in {case1}: {reso} sim.')
plt.colorbar( c )

"""
ax=axs[1]
c = ax.contourf( lon, lat, np.average(rho_epwp_2[:,z,:,:],axis=0) ,levels=flev, cmap=cmap , norm=norm, extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
ax.set_title( f'Resolved GW {r"$\tau$"} in {case2}: {reso} sim.')
plt.colorbar( c )
"""
plt.suptitle( title_ , fontsize=24) 

#print(X1.lev[z].values)
print(zlev[z])


In [ ]:
print( z )
plt.plot( zo_1[0,z,:,:].flatten() )


In [ ]:
z0,lat0,lon0 =23_000., -50. , 30. 
z_event  = np.argmin( np.abs( zlev - z0 ) ) 
print( zlev[ z_event ] )

event_list=[]

for t in np.arange( nt ):
    #events  = auti.find_gw_events(epwp= rho_epwp_1[t,z,:,:] , thresh=0.01, connectivity=8)
    events = auti.find_gw_events_watershed(epwp= rho_epwp_1[t,z,:,:] , thresh=0.02 )  #, thresh=0.01, connectivity=8)
    event_list.append( events )

# Not useful in current form
#tracks = auti.track_events(event_lists=event_list , max_dist=5)

In [ ]:
import pandas as pd
#import xarray as xr

event_id = 0
records = []

for t, events in enumerate(event_list):
    for ev in events:
        records.append({
            "event_id": event_id,
            "itime": int(t),
            "iy": int(ev["iy"]),
            "ix": int(ev["ix"]),
            "epwp_max": float(ev["epwp_max"]),
            "size": int(ev["size"]),
            "epwp_sum": float(ev["epwp_sum"]),
        })
        event_id += 1

df = pd.DataFrame(records)

ds = xr.Dataset.from_dataframe(df)

ds = ds.assign_coords(
    lat=("lat", lat),
    lon=("lon", lon),
    zlev=("zlev", zlev),
)

ds.attrs["description"] = "GW events from resolved momentum flux"
ds.attrs["epwp_definition"] = "sqrt(upwp^2 + vpwp^2)"
ds.attrs["threshold"] = 0.02
ds.attrs["vertical_level"] = z_event
ds.attrs["source_files"] = f"{A1.base_file_name}.%y-%m-%d-%s.nc"
ds.attrs["start_date"] = f"{A1.start_date}"
ds.attrs["step_size_in_hours"] = f"{A1.step_size}"



ds["lat_event"] = ("event", lat[ds["iy"]])
ds["lon_event"] = ("event", lon[ds["ix"]])
#ds["time_event"] = ("event", ds["itime"]*A1.step_size)


In [ ]:
A1['case']=case1
thresh=0.02

In [ ]:
outfile=f"{A1.case}_Events_epwp{thresh:.0e}_Z{0.001*zlev[z_event]:.0f}km_{A1.start_date}_{A1.end_date}.nc"
print( outfile )

In [ ]:
ds.to_netcdf( outfile )

In [ ]:
time_ev = ds.itime.values
lat_ev = ds.lat_event.values
lon_ev = ds.lon_event.values
epwp_ev = ds.epwp_max.values

In [ ]:
print(time_ev[0:19+25+1])
print(len(event_list[1]))

In [ ]:
#plt.scatter( lon_ev, lat_ev  )
plt.scatter(
    lon_ev,
    lat_ev,
    c=np.log10(epwp_ev),
    s=10,
)
plt.colorbar(label="epwp_max")
plt.xlabel("lon")
plt.ylabel("lat")

In [ ]:
print( len(event_list) )
event_list[0]

In [ ]:
######################################################
# Checks events found for a particular time 
######################################################
t0=10
iy2 = [ev["iy"] for ev in event_list[t0] ]
ix2 = [ev["ix"] for ev in event_list[t0] ]

trunc_k, sigma=10, 4.5
#trunc_k, sigma=20, 7.5

htopo_sm = gaussian_filter(  htopo , sigma=sigma,
                           truncate=((trunc_k-1)/2)/sigma, mode="nearest")

mask=htopo_sm <= 1.0

z=np.argmin( np.abs( zlev - 23_000. ) )

title_ = f'Z={zlev[z]/1000.:.0f} km '     #        camsnap-yaaaa.h.2004-06-15-21600.nc'


#flev=[0.0001,0.0002,0.0005,  0.001,0.002,0.005,  0.01,0.02,0.05,  0.1,0.2,0.5,   1.0  ]
flev=[0.0001,0.0002,0.0005,  0.001,0.0015,0.002,0.003,0.004,0.005,0.006,0.008,    0.01,0.015,0.02,0.03,0.04,0.05,0.06,  0.1,0.2,0.5,   1.0  ]
flev=0.1*np.asarray( flev )
print(flev)
cmapN='gist_ncar'

cmap = plt.cm.bwr  #gist_ncar  # Or any other colormap
cmap = plt.cm.gist_ncar # .plasma #gist_ncar  #gist_ncar  # Or any other colormap
norm = mcolors.BoundaryNorm(boundaries=flev, ncolors=cmap.N, clip=False)

topocolor='white' #'black'
topocolor='black'


fig,axs=plt.subplots( 1, 2, figsize=(30,6) )

t=t0

ax=axs[0]
c = ax.contourf( lon, lat, rho_epwp_1[t,z,:,:]  ,levels=flev, cmap=cmap , norm=norm, extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
eve2 = ax.scatter( lon[ix2] , lat[iy2], color='black', marker='+', s=80)


ax.set_title( f'Resolved GW {r"$\tau$"} in {case1}: {reso} sim.')
plt.colorbar( c )

ax=axs[1]
c = ax.contourf( lon, lat, rho_epwp_1[t,z,:,:]*mask,levels=flev, cmap=cmap , norm=norm, extend='both'  )
#c = ax.contourf( lon, lat, np.average(epwp_1[:,z,:,:],axis=0) ,levels=21, cmap=cmap , extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
ax.set_title( f'Resolved GW {r"$\tau$"} in {case1}: {reso} sim.')
plt.colorbar( c )

"""
ax=axs[1]
c = ax.contourf( lon, lat, np.average(rho_epwp_2[:,z,:,:],axis=0) ,levels=flev, cmap=cmap , norm=norm, extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)
ax.set_title( f'Resolved GW {r"$\tau$"} in {case2}: {reso} sim.')
plt.colorbar( c )
"""
plt.suptitle( title_ , fontsize=24) 

#print(X1.lev[z].values)
print(zlev[z])


In [ ]:
count=0
for evs in event_list:
    for ev in evs:
        lat0=lat[ ev['iy'] ]
        lon0=lon[ ev['ix'] ]
        if (lat0>=-60) and (lat0<=-40.) and (lon0 >= 0.) and (lon0 <= 60.):
            count=count+1

print( count )

u_comp = np.zeros(( count, nz ))
v_comp = np.zeros(( count, nz ))
zeta_comp = np.zeros(( count, nz ))
epwp_comp = np.zeros(( count, nz ))
time_ev = np.zeros(( count ))
iy_ev = np.zeros(( count ))
ix_ev = np.zeros(( count ))


c,t=0,0
for evs in event_list:
    for ev in evs:
        lat0=lat[ ev['iy'] ]
        lon0=lon[ ev['ix'] ]
        if (lat0>=-60) and (lat0<=-40.) and (lon0 >= 0.) and (lon0 <= 60.):
            epwp_comp[c,:] = rho_epwp_1[t,:, ev['iy'], ev['ix'] ]
            u_comp[c,:] = u_1[t,:, ev['iy'], ev['ix'] ]
            zeta_comp[c,:] = zeta_1[t,:, ev['iy'], ev['ix'] ]
            time_ev[c] = t
            iy_ev[c],ix_ev[c] = ev['iy'], ev['ix']
            c=c+1
    t=t+1

print( c )

In [ ]:
importlib.reload( auti )


In [ ]:
window=[3,2,2]
zeta_4D = auti.composite4D( event_list=event_list, aa=zeta_1 , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=[-60,-40], lon_range=[0,60] )
epwp_4D = auti.composite4D( event_list=event_list, aa=rho_epwp_1 , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=[-60,-40], lon_range=[0,60] )
u_4D    = auti.composite4D( event_list=event_list, aa=u_1 , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=[-60,-40], lon_range=[0,60] )
v_4D    = auti.composite4D( event_list=event_list, aa=v_1 , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=[-60,-40], lon_range=[0,60] )


In [ ]:
zeta_MMM = auti.collapseSpace( zeta_4D )

In [ ]:
zeta_4D_c = np.mean( zeta_4D , axis=0 )
epwp_4D_c = np.mean( epwp_4D , axis=0 )
u_4D_c    = np.mean( u_4D , axis=0 )


In [ ]:
from matplotlib.ticker import MaxNLocator

n=100


print( f"time = {time_ev[n]}, lat = {lat[int(iy_ev[n])]}, lon = {lon[int(ix_ev[n])]} " )
#epwp,vort,U,zl = epwp_4D[n,5,:,5,5],zeta_4D[n,4,:,4,6], u_4D[n,4,:,4,6], zlev
epwp,vort,U,zl = epwp_4D[n,3,:,2,2],zeta_4D[n,2,:,1,3], u_4D[n,2,:,1,3], zlev
fig, ax1 = plt.subplots(figsize=(5, 8))

# First variable: upwp
ax1.plot(epwp, zl, color='tab:blue', linewidth=2)
ax1.set_xlabel("upwp", color='tab:blue')
ax1.tick_params(axis='x', colors='tab:blue')
ax1.set_ylabel("z")
ax1.set_ylim(0,40_000.)

# Second variable: U
ax2 = ax1.twiny()
ax2.plot(U, zl, color='tab:orange', linewidth=2, linestyle='--')
ax2.set_xlabel("U", color='tab:orange')
ax2.tick_params(axis='x', colors='tab:orange')

# Third variable: vorticity
ax3 = ax1.twiny()
ax3.spines["top"].set_position(("axes", 1.12))   # move this x-axis upward
ax3.plot(vort, zl, color='tab:green', linewidth=2, linestyle=':')

#for ix in np.arange(11):
#    for iy in np.arange(11):
for it in np.arange(start=0,stop=4):
    #for ix in np.arange(start=3,stop=7):
    #    for iy in np.arange(start=3,stop=7):
    #        alpha=0.05 + it/20.
    #        ax3.plot(  zeta_4D[n,it,:,iy,ix]  , zl, color='tab:green', linewidth=1, linestyle='-', alpha=alpha)
    alpha=0.05 + it/20.
    ax3.plot(  zeta_MMM[0][n,it,:]  , zl, color='tab:green', linewidth=2, linestyle='-', alpha=alpha)
    ax3.plot(  zeta_MMM[1][n,it,:]  , zl, color='tab:green', linewidth=1, linestyle='-', alpha=alpha)
    ax3.plot(  zeta_MMM[2][n,it,:]  , zl, color='tab:green', linewidth=1, linestyle='-', alpha=alpha)

ax3.set_xlabel("vorticity", color='tab:green')
ax3.tick_params(axis='x', colors='tab:green')
ax3.set_xlim(-0.0003,0.0002 )
ax3.xaxis.set_major_locator(MaxNLocator(4))
# Optional: if pressure or model level increases downward
# ax1.invert_yaxis()

plt.tight_layout()


In [ ]:
print( poopypants )

In [ ]:
from matplotlib.ticker import MaxNLocator



flev=[0.0001,0.0002,0.0005,  0.001,0.0015,0.002,0.003,0.004,0.005,0.006,0.008,    0.01,0.015,0.02,0.03,0.04,0.05,0.06,  0.1,0.2,0.5,   1.0  ]
flev=0.1*np.asarray( flev )
print(flev)
cmapN='gist_ncar'

cmap = plt.cm.bwr  #gist_ncar  # Or any other colormap
cmap = plt.cm.gist_ncar # .plasma #gist_ncar  #gist_ncar  # Or any other colormap
norm = mcolors.BoundaryNorm(boundaries=flev, ncolors=cmap.N, clip=False)



n=3


















print( f"time = {time_ev[n]}, lat = {lat[int(iy_ev[n])]}, lon = {lon[int(ix_ev[n])]} " )
lat_0 = lat[int(iy_ev[n])] 
lon_0 = lon[int(ix_ev[n])]
t_0 = int(time_ev[n])

t,z=t_0,z_event
zvo = np.argmin( np.abs( zlev - 10_000. ) ) 

fig, ax = plt.subplots(figsize=(8, 5))
c = ax.contourf( lon, lat, rho_epwp_1[t,z,:,:] ,levels=flev, cmap=cmap , norm=norm, extend='both'  )
zoo = ax.contour( lon, lat, zeta_1[t-0,zvo,:,:] ,levels=21, colors='black' )
#c = ax.contourf( lon, lat, np.average(epwp_1[:,z,:,:],axis=0) ,levels=21, cmap=cmap , extend='both'  )
to = ax.contour( lon, lat, htopo, levels=[1,100] ,colors=topocolor)

#ax.set_xlim( 0,30 )
#ax.set_ylim( -55,-35 )


ax.set_xlim( np.asarray( [lon_0-10., lon_0+10.] ) )
ax.set_ylim( np.asarray( [lat_0-10., lat_0+10.] ) )

"""
# First variable: upwp
ax1.plot(epwp, zl, color='tab:blue', linewidth=2)
ax1.set_xlabel("upwp", color='tab:blue')
ax1.tick_params(axis='x', colors='tab:blue')
ax1.set_ylabel("z")
ax1.set_ylim(0,40_000.)

# Second variable: U
ax2 = ax1.twiny()
ax2.plot(U, zl, color='tab:orange', linewidth=2, linestyle='--')
ax2.set_xlabel("U", color='tab:orange')
ax2.tick_params(axis='x', colors='tab:orange')

# Third variable: vorticity
ax3 = ax1.twiny()
ax3.spines["top"].set_position(("axes", 1.12))   # move this x-axis upward
ax3.plot(vort, zl, color='tab:green', linewidth=2, linestyle=':')
ax3.set_xlabel("vorticity", color='tab:green')
ax3.tick_params(axis='x', colors='tab:green')
ax3.set_xlim(-0.0003,0.0002 )
ax3.xaxis.set_major_locator(MaxNLocator(4))
# Optional: if pressure or model level increases downward
# ax1.invert_yaxis()

plt.tight_layout()
"""

In [ ]:
# Pick locatio and time for a sample plot. he values below were picked for ne240 x02 i.e., Aug 2004

z0,lat0,lon0 =25_000., 12. , 250. 
z0,lat0,lon0 =25_000., -50. , 130. 
z0,lat0,lon0 =25_000., -50. , 30. 
#z0,lat0,lon0 =25_000., -12. , 180. 

t,z,y,x = 43, np.argmin( np.abs( zlev - z0 ) ) , np.argmin( np.abs( lat - lat0 ) ) , np.argmin( np.abs( lon - lon0 ) )


In [ ]:
plt.plot( rho_epwp_1[:,z,y,x] )
plt.xlim(40,50)

In [ ]:

t=43

In [ ]:

plt.plot( np.average(rho_epwp_1[:,:,y,x],axis=0) , zlev )
#plt.plot( np.average(rho_epwp_2[:,1:,y,x],axis=0) , zlev )
#plt.xlim(0,0.01)


In [ ]:
plt.contourf( lat, zlev, np.mean( np.mean( u_1 , axis=3) ,0) )
plt.colorbar()

In [ ]:
nt,nz,ny,nx = np.shape( u_1 )
zeta_1 = np.zeros( ( nt,nz,ny,nx) )

for t in np.arange( nt ):
    for z in np.arange( nz ):
        zeta_1[t,z,:,:] = nuti.Sphere_Curl2( f_x=u_1[t,z,:,:]  , f_y=v_1[t,z,:,:] , lat=lat, lon=lon , wrap=True, verbose=False)


In [ ]:
from matplotlib.ticker import MaxNLocator

epwp,vort,U,zl = rho_epwp_1[t,:,y,x] , zeta_1[t,:,y,x] , u_1[t,:,y,x], zlev
fig, ax1 = plt.subplots(figsize=(5, 8))

# First variable: upwp
ax1.plot(epwp, zl, color='tab:blue', linewidth=2)
ax1.set_xlabel("upwp", color='tab:blue')
ax1.tick_params(axis='x', colors='tab:blue')
ax1.set_ylabel("z")
ax1.set_ylim(0,40_000.)

# Second variable: U
ax2 = ax1.twiny()
ax2.plot(U, zl, color='tab:orange', linewidth=2, linestyle='--')
ax2.set_xlabel("U", color='tab:orange')
ax2.tick_params(axis='x', colors='tab:orange')

# Third variable: vorticity
ax3 = ax1.twiny()
ax3.spines["top"].set_position(("axes", 1.12))   # move this x-axis upward
ax3.plot(vort, zl, color='tab:green', linewidth=2, linestyle=':')
ax3.set_xlabel("vorticity", color='tab:green')
ax3.tick_params(axis='x', colors='tab:green')
ax3.set_xlim(-0.0003,0.0002 )
ax3.xaxis.set_major_locator(MaxNLocator(4))
# Optional: if pressure or model level increases downward
# ax1.invert_yaxis()

plt.tight_layout()


In [ ]:
len(event_list)

In [ ]:
evo=event_list[100]

In [ ]:
len(evo)

In [ ]:
evo[10]

In [ ]:
print( count )

In [ ]:
u_comp = np.zeros(( count, nz ))
v_comp = np.zeros(( count, nz ))
zeta_comp = np.zeros(( count, nz ))
epwp_comp = np.zeros(( count, nz ))


c,t=0,0
for evs in event_list:
    for ev in evs:
        lat0=lat[ ev['iy'] ]
        lon0=lon[ ev['ix'] ]
        #if (lat0>=-60) and (lat0<=-40.) and (lon0 <= 50.):
        if (lat0>=-60) and (lat0<=-40.) and (lon0 >= 190.) and (lon0 <= 270.):
            epwp_comp[c,:] = rho_epwp_1[t,:, ev['iy'], ev['ix'] ]
            u_comp[c,:] = u_1[t,:, ev['iy'], ev['ix'] ]
            zeta_comp[c,:] = zeta_1[t,:, ev['iy'], ev['ix'] ]
            c=c+1
    t=t+1

In [ ]:
plt.plot( np.mean( zeta_comp, axis=0 ), zlev )
for c in np.arange( count ):
    plt.plot( zeta_comp[c,:] , zlev, color='black' , alpha=.33 )

plt.plot( np.mean( zeta_comp, axis=0 ), zlev )

#plt.xlim(0.,.3)
plt.ylim(0.,20000.)


In [ ]:
plt.plot( np.mean( u_comp, axis=0 ), zlev )
for c in np.arange( count ):
    plt.plot( u_comp[c,:] , zlev, color='black' , alpha=.33 )

plt.plot( np.mean( u_comp, axis=0 ), zlev )
plt.ylim(0.,20000.)

#plt.xlim(0.,.3)

In [ ]:
#plt.xlim(0.,.02)